# GNN Edge Dominating Set — Watts-Strogatz (Small-World) Graphs

## 1. Install

In [1]:
!pip install torch torch_geometric networkx numpy pandas scikit-learn tqdm --quiet

## 2. Imports & Config

In [2]:
import random, time, numpy as np, pandas as pd
import networkx as nx, torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm import tqdm

SEED        = 42
N_TRAIN     = 1000
N_TEST      = 200
FEAT_DIM    = 12
HIDDEN      = 128
N_LAYERS    = 4
BATCH_SIZE  = 32
EPOCHS      = 80
LR          = 3e-3
ALPHA       = 1.0
BETA        = 10.0
N_SAMPLES   = 100

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 3. Watts-Strogatz Graph Generator

In [3]:
def make_graph(seed):
    rng = random.Random(seed)
    n   = rng.randint(15, 40)
    k   = rng.choice([4, 6, 8])   # must be even and < n
    k   = min(k, n - 1 if n % 2 == 0 else n - 2)
    p   = rng.uniform(0.05, 0.5)  # rewiring probability
    for attempt in range(30):
        G = nx.watts_strogatz_graph(n, k, p, seed=seed + attempt)
        if nx.is_connected(G) and G.number_of_edges() > 0:
            return nx.convert_node_labels_to_integers(G)
    # Fallback: increase rewiring
    G = nx.watts_strogatz_graph(n, k, 0.5, seed=seed)
    return nx.convert_node_labels_to_integers(G)

G0 = make_graph(0)
print(f'Sample: {G0.number_of_nodes()} nodes, {G0.number_of_edges()} edges')
print(f'Average clustering: {nx.average_clustering(G0):.3f}')

Sample: 27 nodes, 81 edges
Average clustering: 0.558


## 4. EDS Algorithms

In [4]:
def build_covers(G,edges):
    n=len(edges); eidx={}
    for i,(u,v) in enumerate(edges): eidx[(u,v)]=i; eidx[(v,u)]=i
    covers=[set() for _ in range(n)]
    for i,(u,v) in enumerate(edges):
        for w in G.neighbors(u):
            j=eidx.get((u,w),eidx.get((w,u),-1))
            if j>=0: covers[i].add(j)
        for w in G.neighbors(v):
            j=eidx.get((v,w),eidx.get((w,v),-1))
            if j>=0: covers[i].add(j)
        covers[i].add(i)
    return covers

def greedy_eds_label(G):
    edges=list(G.edges()); n=len(edges); eidx={}
    for i,(u,v) in enumerate(edges): eidx[(u,v)]=i; eidx[(v,u)]=i
    def cov(i):
        u,v=edges[i]; c=set()
        for nb in list(G.edges(u))+list(G.edges(v)): c.add(eidx[nb])
        return c
    covered,remaining,sel=set(),set(range(n)),[]
    while covered!=set(range(n)):
        best=max(remaining,key=lambda i:len(cov(i)-covered))
        sel.append(best); covered|=cov(best); remaining.discard(best)
    y=torch.zeros(n)
    for i in sel: y[i]=1.0
    return y

def greedy_eds_size(G):
    edges=list(G.edges()); n=len(edges); eidx={}
    for i,(u,v) in enumerate(edges): eidx[(u,v)]=i; eidx[(v,u)]=i
    def cov(i):
        u,v=edges[i]; c=set()
        for nb in list(G.edges(u))+list(G.edges(v)): c.add(eidx[nb])
        return c
    covered,remaining,sel=set(),set(range(n)),[]
    while covered!=set(range(n)):
        best=max(remaining,key=lambda i:len(cov(i)-covered))
        sel.append(best); covered|=cov(best); remaining.discard(best)
    return len(sel)

def two_approx_size(G): return len(nx.maximal_matching(G))

def is_valid(covers,sel_idx,n):
    covered=set()
    for i in sel_idx: covered|=covers[i]
    return covered==set(range(n))

print('Algorithms ready.')

Algorithms ready.


## 5. Feature Engineering & Line Graph

In [5]:
def edge_features(G,seed=0):
    rng=np.random.default_rng(seed); edges=list(G.edges())
    deg=dict(G.degree()); maxd=max(deg.values()) or 1
    try:    bc=nx.betweenness_centrality(G,normalized=True)
    except: bc={v:0.0 for v in G}
    try:    ec=nx.eigenvector_centrality_numpy(G)
    except: ec={v:0.0 for v in G}
    try:    cl=nx.clustering(G)   # very informative for WS graphs
    except: cl={v:0.0 for v in G}
    rows=[]
    for u,v in edges:
        du,dv=deg[u]/maxd,deg[v]/maxd
        rows.append([du,dv,(du+dv)/2,abs(du-dv),du*dv,
                     min(du,dv)/(max(du,dv)+1e-6),
                     bc[u],bc[v],ec[u],ec[v],cl[u],cl[v]])
    x=np.array(rows,dtype=np.float32)
    x+=rng.normal(0,0.01,x.shape).astype(np.float32)
    return torch.from_numpy(x)

def to_pyg(G,seed=0):
    edges=list(G.edges()); n=len(edges); eidx={}
    for i,(u,v) in enumerate(edges): eidx[(u,v)]=i; eidx[(v,u)]=i
    adj=set()
    for node in G.nodes():
        inc=list(G.edges(node))
        for i in range(len(inc)):
            for j in range(i+1,len(inc)):
                ei,ej=eidx[inc[i]],eidx[inc[j]]
                if ei!=ej: adj.add((ei,ej)); adj.add((ej,ei))
    ei=(torch.tensor(list(adj),dtype=torch.long).t().contiguous()
        if adj else torch.zeros((2,0),dtype=torch.long))
    data=Data(x=edge_features(G,seed),edge_index=ei,y=greedy_eds_label(G),num_nodes=n)
    data.n_nodes=G.number_of_nodes(); data.n_edges=n
    data.eds_size=int(data.y.sum().item())
    return data

print(f'Feature shape: {to_pyg(G0).x.shape}')

Feature shape: torch.Size([81, 12])


## 6. Generate Dataset (1000 Train + 200 Test)

In [6]:
def generate(n,offset,desc):
    return [to_pyg(make_graph(offset+i),seed=offset+i)
            for i in tqdm(range(n),desc=desc)]

t0          = time.time()
train_data  = generate(N_TRAIN,offset=0,            desc='Train')
test_data   = generate(N_TEST, offset=N_TRAIN+5000, desc='Test ')
test_graphs = [make_graph(N_TRAIN+5000+i) for i in range(N_TEST)]

print(f'Done in {time.time()-t0:.1f}s')
print(f'Train edges mean = {np.mean([d.n_edges  for d in train_data]):.1f}')
print(f'Train EDS   mean = {np.mean([d.eds_size for d in train_data]):.1f}')
print(f'Test  edges mean = {np.mean([d.n_edges  for d in test_data]):.1f}')
print(f'Test  EDS   mean = {np.mean([d.eds_size for d in test_data]):.1f}')

Test : 100%|██████████| 200/200 [00:01<00:00, 129.14it/s]

Done in 8.9s
Train edges mean = 81.5
Train EDS   mean = 11.0
Test  edges mean = 84.2
Test  EDS   mean = 11.3


## 7. Model

In [7]:
class ResBlock(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.conv=SAGEConv(ch,ch); self.ln=nn.LayerNorm(ch)
    def forward(self,x,ei): return F.relu(self.ln(self.conv(x,ei)+x))

class EDSNet(nn.Module):
    def __init__(self,in_ch=FEAT_DIM,hidden=HIDDEN,n_layers=N_LAYERS):
        super().__init__()
        self.proj=nn.Linear(in_ch,hidden)
        self.blocks=nn.ModuleList([ResBlock(hidden) for _ in range(n_layers)])
        self.head=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Dropout(0.1),nn.Linear(64,1))
        nn.init.constant_(self.head[-1].bias,-2.0)
    def forward(self,x,ei):
        x=F.relu(self.proj(x))
        for blk in self.blocks: x=blk(x,ei)
        return torch.sigmoid(self.head(x)).squeeze(-1)

def eds_loss(probs,edge_index,alpha=ALPHA,beta=BETA):
    cov=probs.clone()
    if edge_index.shape[1]>0:
        src,dst=edge_index; cov.scatter_add_(0,dst,probs[src])
    return alpha*probs.sum()+beta*F.relu(1.0-cov).sum()

model=EDSNet().to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 142,593


## 8. Train

In [8]:
train_loader=DataLoader(train_data,batch_size=BATCH_SIZE,shuffle=True)
test_loader =DataLoader(test_data, batch_size=BATCH_SIZE,shuffle=False)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
train_losses,test_losses=[],[]

for epoch in range(1,EPOCHS+1):
    model.train(); tl=0.0
    for batch in train_loader:
        batch=batch.to(DEVICE); optimizer.zero_grad()
        p=model(batch.x,batch.edge_index)
        loss=eds_loss(p,batch.edge_index)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step(); tl+=loss.item()
    train_losses.append(tl/len(train_loader))
    model.eval(); vl=0.0
    with torch.no_grad():
        for batch in test_loader:
            batch=batch.to(DEVICE)
            vl+=eds_loss(model(batch.x,batch.edge_index),batch.edge_index).item()
    test_losses.append(vl/len(test_loader)); scheduler.step()
    if epoch%10==0 or epoch==1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  train={train_losses[-1]:.3f}  test={test_losses[-1]:.3f}')

print('Training done.')

Epoch   1/80  train=507.596  test=430.016
Epoch  10/80  train=397.969  test=324.102
Epoch  20/80  train=398.994  test=282.001
Epoch  30/80  train=377.853  test=266.668
Epoch  40/80  train=294.858  test=270.168
Epoch  50/80  train=273.051  test=254.942
Epoch  60/80  train=278.915  test=248.649
Epoch  70/80  train=261.524  test=244.341
Epoch  80/80  train=260.431  test=243.111
Training done.


## 9. Sampling Decoder

In [9]:
def sample_decode_one(covers,scores,n,rng):
    probs=np.clip(scores,1e-6,1-1e-6)
    weights=probs/probs.sum()
    order=rng.choice(n,size=n,replace=False,p=weights)
    covered=set(); sel=[]; all_edges=set(range(n))
    for idx in order:
        if covers[idx]-covered: sel.append(idx); covered|=covers[idx]
        if covered==all_edges: break
    if covered!=all_edges:
        for idx in np.argsort(-scores):
            if covers[idx]-covered: sel.append(idx); covered|=covers[idx]
            if covered==all_edges: break
    return sel

def sampling_decoder(G,edges,scores,n_samples=N_SAMPLES,seed=0):
    covers=build_covers(G,edges); n=len(edges)
    rng=np.random.default_rng(seed)
    best_sel=None; best_size=float('inf')
    for _ in range(n_samples):
        sel=sample_decode_one(covers,scores,n,rng)
        if len(sel)<best_size and is_valid(covers,sel,n):
            best_size=len(sel); best_sel=sel
    if best_sel is None:
        best_sel=[]; covered=set()
        for idx in np.argsort(-scores):
            if covers[idx]-covered: best_sel.append(idx); covered|=covers[idx]
            if covered==set(range(n)): break
    return [edges[i] for i in best_sel], best_sel

print('Sampling decoder ready.')

Sampling decoder ready.


## 10. Evaluate on 200 Test Graphs

In [10]:
model.eval(); records=[]
for i,(G,data) in enumerate(tqdm(zip(test_graphs,test_data),total=N_TEST,desc='Evaluating')):
    edges=list(G.edges()); n_e=len(edges)
    d=data.to(DEVICE)
    with torch.no_grad(): scores=model(d.x,d.edge_index).cpu().numpy()
    gnn_eds,gnn_idx=sampling_decoder(G,edges,scores,n_samples=N_SAMPLES,seed=i)
    gr_size=greedy_eds_size(G); ap_size=two_approx_size(G)
    gnn_lbl=np.zeros(n_e)
    for j in gnn_idx: gnn_lbl[j]=1
    gr_lbl=data.y.numpy()
    records.append(dict(
        n_nodes=G.number_of_nodes(), n_edges=n_e,
        gnn_size=len(gnn_eds), greedy_size=gr_size, approx_size=ap_size,
        gnn_ratio=len(gnn_eds)/max(gr_size,1), ap_ratio=ap_size/max(gr_size,1),
        f1=f1_score(gr_lbl,gnn_lbl,zero_division=0),
        precision=precision_score(gr_lbl,gnn_lbl,zero_division=0),
        recall=recall_score(gr_lbl,gnn_lbl,zero_division=0),
        score_std=float(scores.std()),
    ))

df=pd.DataFrame(records)
print(df[['gnn_size','greedy_size','approx_size','gnn_ratio','f1']].describe().round(3))

Evaluating: 100%|██████████| 200/200 [00:02<00:00, 73.26it/s]

       gnn_size  greedy_size  approx_size  gnn_ratio       f1
count    200.00      200.000      200.000    200.000  200.000
mean      13.40       11.320       13.390      1.179    0.144
std        4.28        3.359        3.916      0.109    0.119
min        6.00        5.000        7.000      0.857    0.000
25%       10.00        9.000       10.000      1.111    0.059
50%       13.00       11.000       13.500      1.182    0.133
75%       17.00       14.000       17.000      1.250    0.216
max       22.00       18.000       20.000      1.500    0.667


## 11. Final Report

In [12]:
print('='*55)
print('  GNN-EDS RESULTS — WATTS-STROGATZ (200 TEST GRAPHS)')
print('='*55)
print(f'  Train/Test        : {N_TRAIN} / {N_TEST}')
print(f'  Sampling passes   : {N_SAMPLES}')
print()
print(f'  EDS size  GNN     : {df["gnn_size"].mean():.2f} ± {df["gnn_size"].std():.2f}')
print(f'  EDS size  Greedy  : {df["greedy_size"].mean():.2f} ± {df["greedy_size"].std():.2f}')
print(f'  EDS size  2-Approx: {df["approx_size"].mean():.2f} ± {df["approx_size"].std():.2f}')
print()
print(f'  GNN / Greedy      : {df["gnn_ratio"].mean():.3f} ± {df["gnn_ratio"].std():.3f}')
print(f'  2A  / Greedy      : {df["ap_ratio"].mean():.3f} ± {df["ap_ratio"].std():.3f}')
print()
print(f'  F1  vs greedy lbl : {df["f1"].mean():.3f}')
print(f'  Precision         : {df["precision"].mean():.3f}')
print(f'  Recall            : {df["recall"].mean():.3f}')
print()
print(f'  Score STD (mean)  : {df["score_std"].mean():.4f}',
      '(ok)' if df['score_std'].mean()>0.05 else 'death spiral?')
print(f'  GNN beats Greedy  : {(df["gnn_ratio"]<1).mean()*100:.1f}% of graphs')
print(f'  GNN beats 2-Approx: {(df["gnn_ratio"]<df["ap_ratio"]).mean()*100:.1f}% of graphs')
print('='*55)

  GNN-EDS RESULTS — WATTS-STROGATZ (200 TEST GRAPHS)
  Train/Test        : 1000 / 200
  Sampling passes   : 100

  EDS size  GNN     : 13.40 ± 4.28
  EDS size  Greedy  : 11.32 ± 3.36
  EDS size  2-Approx: 13.39 ± 3.92

  GNN / Greedy      : 1.179 ± 0.109
  2A  / Greedy      : 1.189 ± 0.105

  F1  vs greedy lbl : 0.144
  Precision         : 0.134
  Recall            : 0.155

  Score STD (mean)  : 0.0372 death spiral?
  GNN beats Greedy  : 1.5% of graphs
  GNN beats 2-Approx: 32.0% of graphs
